In [2]:
import openpyxl
import math
import numpy as np
import pandas as pd

In [19]:
# 最大值測試
luminance = 54 * (255) + 183 * (255) + 19 * (255)

In [10]:
Rw = 54
Gw = 183
Bw = 19
print(type(Rw))
print(Rw.bit_length())
print(Gw.bit_length())
print(Bw.bit_length())

<class 'int'>
6
8
5


In [108]:
print("Decimal luminance:", luminance)
print("binary:", bin(luminance))
luminance.bit_length()

Decimal luminance: 65280
binary: 0b1111111100000000


16

In [17]:
LUT_INDEX_BITS = 12

workbook = openpyxl.Workbook()
sheet = workbook.active
sheet.title = "Lm_base_LUT"

sheet["A1"] = "base 12 bit"
sheet["B1"] = "1.base base value"

steps = 1 << LUT_INDEX_BITS
for i in range(steps):
    x = 1.0 + (i / steps)
    y0 = math.log10(x)

    current_row = i + 2
    sheet.cell(row=current_row, column=1).value = i
    sheet.cell(row=current_row, column=2).value = int(y0 * (1 << 14))

    if (i  == steps - 1):
        print("i: ", i)
        print("x: ", x)
        print("y0: ", y0)
        print("y0(Q.14): ", y0 * (1 << 14))
        print(int(y0 * (1 << 14)).bit_length())

workbook.save("./Lm_base_LUT.xlsx")

i:  4095
x:  1.999755859375
y0:  0.30097697796484885
y0(Q.14):  4931.2068069760835
13


In [102]:
def load_lut_from_excel(file_path, input_col, output_col):
    """
    讀取 Excel 並回傳輸入(X)與輸出(Y)的對照陣列。
    """
    try:
        df = pd.read_excel(file_path)
        if input_col not in df.columns or output_col not in df.columns:
            print(f"錯誤: 檔案 {file_path} 中找不到欄位 '{input_col}' 或 '{output_col}'")
            return None, None
        # 確保數據按輸入值由小到大排序 (np.interp 需要排序過的 X)
        df = df.sort_values(by=input_col)
        
        lut_x = df[input_col].values
        lut_y = df[output_col].values
        
        print(f"LUT 載入成功。範圍: {lut_x.min()} ~ {lut_x.max()}, 點數: {len(lut_x)}")
        print(f"LUT Max Output: {lut_y.max()}, Min Output: {lut_y.min()}")
        return lut_x, lut_y
    except Exception as e:
        print(f"讀取 LUT 失敗: {e}")
        return None, None

In [ ]:
Lm_LUT = "./Lm_base_LUT.xlsx"

lut_x_l, lut_y_l = load_lut_from_excel(Lm_LUT, input_col="base 12 bit", output_col="1.base base value")

LUT 載入成功。範圍: 0 ~ 4095, 點數: 4096
LUT Max Output: 4931, Min Output: 0


In [104]:
print(math.log10(2))

LOG2_CONST = round(math.log10(2) * (1 << 14))
print(LOG2_CONST * 2 ** -14)
print(LOG2_CONST) # need 13 bit unsigned

0.3010299956639812
0.301025390625
4932


In [18]:
255*4932*2**-14

76.761474609375

In [105]:
def run(r, g, b, e, bias=128):
    lum = 54 * (r) + 183 * (g) + 19 * (b)
    true_lim = lum / (256 * 256) * 2 ** (e-128)
    print(lum / (256 * 256) * 2 ** (e-128))
    print(bin(lum))
    msb = lum.bit_length() - 1
    print(msb)
    TARGET_MSB = 15
    shift = TARGET_MSB - msb
    reg = lum << shift
    print(bin(reg))
    idx = (reg >> (TARGET_MSB - 12)) & 0xFFF
    print(bin(idx), idx)
    base = lut_y_l[idx]
    print(base)

    exp_val = (e - bias) + msb - 16
    print("exp_val: ", exp_val)
    print("LOG2_CONST: ", LOG2_CONST)
    exp_log = exp_val * LOG2_CONST
    print(exp_log)

    log_sum = base + exp_log
    log_sum_fp32 = log_sum * 2 ** -14
    print(log_sum, log_sum_fp32)
    print(math.log10(true_lim))
    print(math.log10(true_lim) - log_sum_fp32)
    print(10**log_sum_fp32)

In [106]:
run(100, 150, 200, 110)

2.1333107724785805e-06
0b1000111100101010
15
0b1000111100101010
0b111100101 485
796
exp_val:  -19
LOG2_CONST:  4932
-93708
-92912 -5.6708984375
-5.670945873598214
-4.7436098213893274e-05
2.133543797465861e-06
